# Jacobian lens — walkthrough

## 0. Setup

Load a model, load a pre-fitted Jacobian lens from the Hub, apply it to a prompt, and render the interactive slice visualisation.

In [1]:
!git clone https://github.com/anthropics/jacobian-lens.git
%cd jacobian-lens

!pip install -e .

c:\Users\jason\CS\jacobian-lens\jacobian-lens


fatal: destination path 'jacobian-lens' already exists and is not an empty directory.


Obtaining file:///C:/Users/jason/CS/jacobian-lens/jacobian-lens
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for jlens (pyproject.toml): started
  Building editable for jlens (pyproject.toml): finished with status 'done'
  Created wheel for jlens: filename=jlens-0.1.0-0.editable-py3-none-any.whl size=8957 sha256=67adde9816aafe68d2d59965c1fd9080761aec5ae3882a24945ba938256725d0
  Stored in directory: C:\Users\jason\AppData\Local\Temp\pip-ephem-wheel-cache-bhu3kyva\wheels\3d\07\90\005f77be2ccbe6152c6e28f571c9f23bba104d3e12e9eb

In [2]:
import torch
print(torch.__version__)
print(torch.__file__)

import jlens
jlens.configure_logging()

MODEL_NAME = "Qwen/Qwen3.5-4B"
# MODEL_NAME = "Qwen/Qwen3.6-27B"

LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "qwen-n1000"
LENS_FILE = {
    "Qwen/Qwen3.5-4B": "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
    "Qwen/Qwen3.6-27B": "qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt",
}[MODEL_NAME]

2.14.0+xpu
c:\Users\jason\CS\jacobian-lens\.venv\Lib\site-packages\torch\__init__.py


## 1. Load the model

`jlens.from_hf` wraps an already-loaded HuggingFace model into `LensModel` interface. (Note: .to("xpu") is used to run Qwen model on Intel Arc B580. Nvidia GPU's should use .cuda() instead.)

In [3]:
import torch
import transformers

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16
).to("xpu")

print(torch.xpu.memory_allocated())

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)
model

c:\Users\jason\CS\jacobian-lens\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 426/426 [00:00<00:00, 3248.09it/s]


8512194560


HFLensModel(Qwen3_5ForCausalLM, n_layers=32, d_model=2560)

## 2. Load a pre-fitted lens

`JacobianLens.from_pretrained` pulls a `.pt` from the Hub (or a local path). The lens holds one `[d_model, d_model]` matrix per layer.

In [4]:
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION
)
lens

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 394.83it/s]


JacobianLens(d_model=2560, n_prompts=1000, source_layers=[0..30] (31 layers))

## 3. Apply: J-lens vs logit lens

`lens.apply(model, prompt, positions=...)` runs one forward pass, transports each layer's residual into the final-layer basis with `J_l`, and decodes through the model's own unembedding. `use_jacobian=False` skips the transport — that's the vanilla logit lens.

Below: a two-hop factual question, read out at the boot token. The J-lens surfaces interpretable tokens at layers where the logit lens is still noise.

In [5]:
prompt = "Fact: The currency used in the country shaped like a boot is"
layers = [
    model.n_layers // 4,
    model.n_layers // 2,
    model.n_layers // 4 * 3,
    model.n_layers - 2,
]

jlens_logits, model_logits, _ = lens.apply(model, prompt, layers=layers, positions=[-2])
logit_lens, _, _ = lens.apply(
    model, prompt, layers=layers, positions=[-2], use_jacobian=False
)


def top5(logits):
    return [tokenizer.decode([t]) for t in logits.topk(5).indices]


for layer in layers:
    print(f"L{layer:>3} logit-lens: {top5(logit_lens[layer][0])}")
    print(f"L{layer:>3} J-lens:     {top5(jlens_logits[layer][0])}")
print(f"model:           {top5(model_logits[0])}")

L  8 logit-lens: ['oman', 'edom', 'ולי', ' Urlaubs', 'GPC']
L  8 J-lens:     [' `', ' boots', ' *', ' `\\', ' heel']
L 16 logit-lens: ['shaw', 'วย', 'amaz', 'REA', '举世']
L 16 J-lens:     ['?', '？', "'?", ' Italy', '____']
L 24 logit-lens: ['的形状', '形状的', '形状', 'shape', '-shaped']
L 24 J-lens:     ['-shaped', ' shape', ' shaped', 'shape', '形状']
L 30 logit-lens: [' is', ' shape', '-shaped', ' shaped', ' heel']
L 30 J-lens:     [' is', ' shape', ' shaped', '-shaped', ' heel']
model:           [' is', ' in', ' on', '.', ' with']


### Cleanup

Working with only 12GB VRAM, so cleaning up after running lens.apply() may be helpful.

In [8]:
import gc
del jlens_logits, model_logits
gc.collect()
torch.xpu.empty_cache()
print(torch.xpu.memory_allocated())

8512195072


## 4. Parsing data and creating slice-vis page

Retrieving top-down-summoning.json and collecting cursory information from its prompts

In [11]:
import json
from pathlib import Path

def load_top_down_summoning_stimuli(data_dir="data/experiments"):
    obj = json.loads((Path(data_dir) / "top-down-summoning.json").read_text(encoding="utf-8"))
    items = obj["items"]
    prompts = [item["stimulus"] for item in items]
    return prompts, items  # keep items around — you'll want key/expected/foil later for scoring

prompts, items = load_top_down_summoning_stimuli()
print(f"{len(prompts)} prompts")
for p in prompts:
    print(len(p.split()), "words —", p)

7 prompts
27 words — She had organised the samples by shade and labelled each one before showing him the options. He recognised the effort, but red had never been his favourite
27 words — So yeah, we'd been hanging out at Jake's place all afternoon, just chilling, when his mom totally freaked out about the mess. Honestly it wasn't even that
25 words — She sealed the final box and stacked it by the door as the light faded. Then she sat down at the empty table and carefully
19 words — The committee had argued through lunch without reaching agreement. When the chair finally called for a vote, the members
18 words — The committee debated for three hours before reaching a verdict. In the end, they declared the proposal utterly
18 words — The committee debated for three hours before reaching a verdict. In the end, they declared the proposal utterly
30 words — The letter arrived on a grey morning, and she read it twice before setting it down. The house felt emptier than it had in years, and e

In [12]:
for item in items:
    ids = model.encode(item["stimulus"], max_length=512)
    print(item["key"], ids.shape[-1], "tokens")

spelling 29 tokens
register 35 tokens
tense 26 tokens
number 21 tokens
pos 20 tokens
pos_tense 20 tokens
tone 34 tokens


Chunks should hopefully prevent out-of-memory errors.

In [13]:
import gc
import torch
from jlens.fitting import valid_position_mask

def run_full_grid_chunked(model, lens, prompt, position_chunk_size=8, max_seq_len=512):
    input_ids = model.encode(prompt, max_length=max_seq_len)
    seq_len = input_ids.shape[-1]
    layers = list(range(model.n_layers))

    valid_mask = valid_position_mask(seq_len)
    positions = [p for p in range(seq_len) if valid_mask[p]]
    if not positions:
        return None  # too short — see the length check above

    jlens_chunks = {layer: [] for layer in layers}
    model_chunks = []

    for start in range(0, len(positions), position_chunk_size):
        chunk = positions[start:start + position_chunk_size]
        jlens_logits, model_logits, _ = lens.apply(
            model, prompt, layers=layers, positions=chunk, max_seq_len=max_seq_len
        )
        for layer in layers:
            jlens_chunks[layer].append(jlens_logits[layer].detach().cpu())
        model_chunks.append(model_logits.detach().cpu())

        del jlens_logits, model_logits
        gc.collect()
        torch.xpu.empty_cache()

    jlens_full = {layer: torch.cat(chunks, dim=1) for layer, chunks in jlens_chunks.items()}  # VERIFY dim=1 is the position axis for your apply() output shape
    model_full = torch.cat(model_chunks, dim=1)
    return jlens_full, model_full, positions